# BYOB MCQ: build your own multiple-choice benchmark

This notebook walks you through the **Build Your Own Benchmark (BYOB)** flow for **multiple-choice questions (MCQ)**. Starting from your source text (or a configured Hugging Face dataset), it **prepares a seed set**, then runs a **staged pipeline**: draft questions with an LLM, judge and refine them, remove near-duplicates, optionally expand or check coverage of wrong answers, validate distractors, flag semantic outliers, filter for hallucination and difficulty, and finally export a **`benchmark.parquet`** you can review or use like other MMLU-style benchmarks.

You work **from top to bottom**: run the setup cell, then **Section 1** to materialize the dataset and write seed data, then **Section 2** in order. Each later stage reads the previous stage’s output via `last_output_path` (a chain of Parquet files under your experiment’s `stage_cache`).

**Prerequisites**

1. **Dependencies:** run the next code cell once. It installs Nemotron from the repo `pyproject.toml` in editable mode with the **`[all]`** optional extras (includes NeMo Curator, Data Designer, OpenAI client, `cosmos-xenna`, etc.). That can take several minutes and may require a CUDA-capable environment for some Curator features.
2. **Input layout:** one or more `.txt` files per subject in the folder structure your config expects (see `input_dir` in `default.yaml`; default is `./assets`, e.g. `assets/wikipedia_finance_india/`). You can build that tree from Wikipedia using `download_wikipedia_data.ipynb` in this folder, or point `input_dir` at your own data.
3. **Config:** edit **`default.yaml`** next to this notebook—especially `output_dir` (default `./assets/output` for all BYOB stage outputs under `<expt_name>/`), `metadata_file`, `prompt_config`, and optional flags such as `do_distractor_expansion` and `do_coverage_check` to match your environment and how heavy you want the run to be.

In [ ]:
# Install Nemotron from repo pyproject.toml (editable) with [all] extras:
# data-designer, nemo-curator, openai, cosmos-xenna, cloud clients, etc.
# Restart the kernel after this cell if imports still fail.
import subprocess
import sys
from pathlib import Path

NEMOTRON_REPO_ROOT = Path.cwd().resolve().parent.parent
_pyproject = NEMOTRON_REPO_ROOT / "pyproject.toml"
if not _pyproject.is_file():
    raise FileNotFoundError(
        f"Could not find pyproject.toml at {_pyproject}. "
        "Set the Jupyter cwd to use-case-examples/build-your-own-benchmark "
        f"(current cwd: {Path.cwd().resolve()})."
    )

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "-e",
        f"{NEMOTRON_REPO_ROOT}[all]",
    ]
)
print("OK:", NEMOTRON_REPO_ROOT, "installed in editable mode with [all]")

Obtaining file:///home/anushak/Downloads/nemotron/Nemotron
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached tomlkit-0.14.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached langdetect-1.0.9-py3-none-any.whl
  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
  Using cached chardet-5.2.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata

In [5]:
import logging
import os
import sys
from pathlib import Path

import pandas as pd

EXAMPLE_DIR = Path.cwd().resolve()
NEMOTRON_REPO_ROOT = EXAMPLE_DIR.parent.parent
NEMOTRON_SRC = NEMOTRON_REPO_ROOT / "src"
CONFIG_PATH = EXAMPLE_DIR / "default.yaml"

if not (NEMOTRON_SRC / "nemotron").is_dir():
    raise FileNotFoundError(
        f"Expected Nemotron `src` at {NEMOTRON_SRC}. "
        f"Set kernel cwd to `use-case-examples/build-your-own-benchmark` (current: {EXAMPLE_DIR})."
    )

if str(NEMOTRON_SRC) not in sys.path:
    sys.path.insert(0, str(NEMOTRON_SRC))

from nemotron.data_prep.stages.byob.config import ByobConfig
from nemotron.data_prep.stages.byob.data_designer_utils import batched_run
from nemotron.data_prep.stages.byob.dataset import make_from_config
from nemotron.data_prep.stages.byob.mcq.utils import (
    prepare_generation_seed_dataset,
    postprocess_generated_questions,
    prepare_judgement_seed_dataset,
    postprocess_judged_questions,
    prepare_filtering_seed_dataset,
    postprocess_filtered_questions,
    prepare_distractor_expansion_seed_dataset,
    postprocess_distractor_expansion,
    prepare_distractor_validity_seed_dataset,
    postprocess_distractor_validity,
)
from nemotron.data_prep.stages.byob.mcq.stages import (
    generate_questions,
    judge_questions,
    expand_distractors,
    check_distractor_validity,
    filter_questions,
)
from nemotron.data_prep.stages.byob.mcq.deduplication import TextSemanticDeduplicationMCQ
from nemotron.data_prep.stages.byob.mcq.text_coverage import TextCoverageMCQ
from nemotron.data_prep.stages.byob.mcq.semantic_outlier import TextSemanticOutlierDetectionMCQ

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing config YAML: {CONFIG_PATH}")

ModuleNotFoundError: No module named 'data_designer'

## 1. Prepare data

Loads the YAML, builds the Hugging Face-backed dataset, samples rows, and writes seed artifacts on disk. Run this before Section 2 so generation has inputs.

In [ ]:
def run_prepare_data(config_path: Path | str) -> None:
    """Parse config, materialize the source dataset, then sample and export the generation seed."""
    config = ByobConfig.from_yaml(str(config_path))
    dataset = make_from_config(config)
    seed = dataset.sample_and_dump()
    print(seed)


run_prepare_data(CONFIG_PATH)

## 2. Generate questions

Run the cells in this section **top to bottom**. The first cell loads `ByobConfig` and defines every parquet path; later cells chain using `last_output_path`.

**Optional** stages are gated in YAML (`do_distractor_expansion`, `do_coverage_check`). When disabled, the notebook skips heavy work and leaves `last_output_path` pointing at the previous stage’s file.

In [ ]:
config = ByobConfig.from_yaml(str(CONFIG_PATH))


def _stage_cache(*parts: str) -> str:
    return os.path.join(config.output_dir, config.expt_name, "stage_cache", *parts)


output_path_generation = _stage_cache("generated_questions.parquet")
output_path_judgement = _stage_cache("judged_questions.parquet")
output_path_semantic_deduplication = _stage_cache("semantic_deduplicated_questions.parquet")
output_path_distractor_expansion = _stage_cache("expanded_distractors.parquet")
output_path_coverage = _stage_cache("coverage_check.parquet")
output_path_distractor_validity = _stage_cache("valid_distractors.parquet")
output_path_semantic_outlier_detection = _stage_cache("semantic_outlier_detection.parquet")
output_path_filtering = _stage_cache("filtered_questions.parquet")
output_path_raw = os.path.join(config.output_dir, config.expt_name, "benchmark_raw.parquet")
output_path_final = os.path.join(config.output_dir, config.expt_name, "benchmark.parquet")

os.makedirs(_stage_cache(), exist_ok=True)

# Updated after each stage; the next cell reads this parquet.
last_output_path: str | None = None

logger.info("Experiment: %s | output_dir: %s", config.expt_name, config.output_dir)

### 2.1 Question generation (required)

Calls the configured LLM through NeMo Data Designer to produce draft multiple-choice questions from the prepared seed. Output: `generated_questions.parquet`.

In [ ]:
logger.info("Generating questions...")
seed_df_generation = prepare_generation_seed_dataset(config)
dataset_out = batched_run(
    generate_questions, config, seed_df_generation, batch_size=config.ndd_batch_size
)
dataset_out.dropna(inplace=True)
dataset_out = postprocess_generated_questions(dataset_out)
dataset_out.to_parquet(output_path_generation)
logger.info("Saved: %s", output_path_generation)
last_output_path = output_path_generation

### 2.2 Judgement (required)

A separate model pass reviews or adjusts the drafts for quality. Input is generation output; result: `judged_questions.parquet`.

In [ ]:
logger.info("Judging questions...")
dataset_in = pd.read_parquet(last_output_path)
seed_df_judgement = prepare_judgement_seed_dataset(config, dataset_in)
dataset_out = batched_run(
    judge_questions, config, seed_df_judgement, batch_size=config.ndd_batch_size
)
dataset_out = postprocess_judged_questions(dataset_in, dataset_out)
dataset_out.to_parquet(output_path_judgement)
logger.info("Saved: %s", output_path_judgement)
last_output_path = output_path_judgement

### 2.3 Semantic deduplication (required)

Clusters near-duplicate questions with sentence embeddings (see `semantic_deduplication_config` in the YAML). Output: `semantic_deduplicated_questions.parquet`.

In [ ]:
logger.info("Semantic deduplication...")
dataset_in = pd.read_parquet(last_output_path)
deduper = TextSemanticDeduplicationMCQ(config)
dataset_out = deduper.run(dataset_in)
dataset_out.to_parquet(output_path_semantic_deduplication)
logger.info("Saved: %s", output_path_semantic_deduplication)
last_output_path = output_path_semantic_deduplication

### 2.4 Distractor expansion (optional)

Runs only when **`do_distractor_expansion: true`** in the YAML. Uses an LLM to grow or refine wrong answer choices. If disabled, this cell does nothing except log a message and **keeps** `last_output_path` on the deduplication file.

In [ ]:
if config.do_distractor_expansion:
    logger.info("Expanding distractors...")
    dataset_in = pd.read_parquet(last_output_path)
    seed_df = prepare_distractor_expansion_seed_dataset(config, dataset_in)
    dataset_out = batched_run(
        expand_distractors, config, seed_df, batch_size=config.ndd_batch_size
    )
    dataset_out = postprocess_distractor_expansion(dataset_in, dataset_out)
    dataset_out.to_parquet(output_path_distractor_expansion)
    logger.info("Saved: %s", output_path_distractor_expansion)
    last_output_path = output_path_distractor_expansion
else:
    logger.info(
        "Distractor expansion disabled (`do_distractor_expansion: false`); using %s",
        last_output_path,
    )

### 2.5 Coverage check (optional)

Runs only when **`do_coverage_check: true`**. Measures how well generated questions cover the source text windows (`coverage_check_config`). If disabled, `last_output_path` stays on the file from 2.3 or 2.4.

In [ ]:
if config.do_coverage_check:
    logger.info("Coverage check...")
    dataset_in = pd.read_parquet(last_output_path)
    text_coverage = TextCoverageMCQ(config)
    dataset_out = text_coverage.analyze(dataset_in)
    dataset_out.to_parquet(output_path_coverage)
    logger.info("Saved: %s", output_path_coverage)
    last_output_path = output_path_coverage
else:
    logger.info(
        "Coverage check disabled (`do_coverage_check: false`); using %s",
        last_output_path,
    )

### 2.6 Distractor validity (required)

LLM checks that distractors are plausible wrong answers. Output: `valid_distractors.parquet`.

In [ ]:
logger.info("Distractor validity...")
dataset_in = pd.read_parquet(last_output_path)
seed_df = prepare_distractor_validity_seed_dataset(config, dataset_in)
dataset_out = batched_run(
    check_distractor_validity, config, seed_df, batch_size=config.ndd_batch_size
)
dataset_out = postprocess_distractor_validity(dataset_in, dataset_out)
dataset_out.to_parquet(output_path_distractor_validity)
logger.info("Saved: %s", output_path_distractor_validity)
last_output_path = output_path_distractor_validity

### 2.7 Semantic outlier detection (required)

Flags choices that are embedding-space outliers relative to the others (`semantic_outlier_detection_config`). Can drop rows when `remove_outliers` is true.

In [ ]:
logger.info("Semantic outlier detection...")
dataset_in = pd.read_parquet(last_output_path)
detector = TextSemanticOutlierDetectionMCQ(config)
dataset_out = detector.detect(dataset_in)
dataset_out.to_parquet(output_path_semantic_outlier_detection)
logger.info("Saved: %s", output_path_semantic_outlier_detection)
last_output_path = output_path_semantic_outlier_detection

### 2.8 Hallucination and easiness filtering (required)

Swarm of models (`filtering_model_configs`) labels hallucination and difficulty, then post-processing merges scores. Output: `filtered_questions.parquet`.

In [ ]:
logger.info("Hallucination / easiness filtering...")
dataset_in = pd.read_parquet(last_output_path)
seed_df = prepare_filtering_seed_dataset(dataset_in, config)
dataset_out = batched_run(
    filter_questions, config, seed_df, batch_size=config.ndd_batch_size
)
dataset_out = postprocess_filtered_questions(dataset_in, dataset_out, config)
dataset_out.to_parquet(output_path_filtering)
logger.info("Saved: %s", output_path_filtering)
last_output_path = output_path_filtering

### 2.9 Final export (required)

Writes `benchmark_raw.parquet`, optionally drops hallucinated or easy rows per YAML, renames columns to MMLU-Pro-style names, and saves **`benchmark.parquet`**. Plan human review on that file before using it in production.

In [ ]:
dataset_final = pd.read_parquet(last_output_path)
dataset_final.to_parquet(output_path_raw)
logger.info("Raw benchmark: %s", output_path_raw)

if config.remove_hallucinated:
    hallucinated_count = int((dataset_final["is_hallucination"] == True).sum())
    dataset_final = dataset_final[dataset_final["is_hallucination"] == False]
    logger.info("Removed %s hallucinated rows", hallucinated_count)
if config.remove_easy:
    easy_count = int((dataset_final["is_easy"] == True).sum())
    dataset_final = dataset_final[dataset_final["is_easy"] == False]
    logger.info("Removed %s easy rows", easy_count)

if len(dataset_final) == 0:
    logger.error("No rows left after filtering; check YAML thresholds and inputs.")
else:
    dataset_final = dataset_final[
        [
            "id_question",
            "question_generated",
            "choices_generated",
            "answer_generated",
            "target_subject",
        ]
    ]
    dataset_final.rename(
        columns={
            "id_question": "question_id",
            "question_generated": "question",
            "choices_generated": "options",
            "answer_generated": "answer",
            "target_subject": "category",
        },
        inplace=True,
    )
    dataset_final["answer_index"] = dataset_final["answer"].apply(lambda x: ord(x) - ord("A"))
    dataset_final["cot_content"] = "-"
    dataset_final["src"] = "-"
    dataset_final.to_parquet(output_path_final)
    logger.info("Final benchmark: %s", output_path_final)
    logger.info("Do human-in-the-loop review before relying on: %s", output_path_final)